---
## 1. Imports & Configuration

In [1]:
import os
import math
import random
from collections import Counter, defaultdict

import pyarrow.parquet as pq
from tqdm import tqdm

# ── Paths ──────────────────────────────────────────────────────────────────
PARQUET_DIR = os.path.join("..", "Assignment_1", "data", "output")
FIRST_SHARD = os.path.join(PARQUET_DIR, "indiccorp_te_tokenized_part-00001.parquet")

# ── Parameters ─────────────────────────────────────────────────────────────
NUM_SENTENCES = 1_000_000
DEV_SIZE      = 1_000
TEST_SIZE     = 1_000
TRAIN_SIZE    = NUM_SENTENCES - DEV_SIZE - TEST_SIZE   # 998,000
SEED          = 42

# Special tokens
SOS = "<s>"
EOS = "</s>"
UNK = "<UNK>"

print(f"Will load {NUM_SENTENCES:,} sentences from:\n  {os.path.abspath(FIRST_SHARD)}")
print(f"Split: train={TRAIN_SIZE:,}  dev={DEV_SIZE:,}  test={TEST_SIZE:,}")

Will load 1,000,000 sentences from:
  d:\Assignments_sem_5\NLP\Assignment_1\data\output\indiccorp_te_tokenized_part-00001.parquet
Split: train=998,000  dev=1,000  test=1,000


---
## 2. Load Data & Train / Dev / Test Split

Same loading and splitting procedure as Question 1 for consistency.

In [2]:
# Load parquet data
table = pq.read_table(FIRST_SHARD, columns=["tokenized_sentence"])
raw_sentences = table.column("tokenized_sentence").to_pylist()

# Tokenize
sentences = [s.split() for s in raw_sentences[:NUM_SENTENCES]]

# Shuffle with same seed as Q1 for identical splits
random.seed(SEED)
random.shuffle(sentences)

train_sentences = sentences[:TRAIN_SIZE]
dev_sentences   = sentences[TRAIN_SIZE : TRAIN_SIZE + DEV_SIZE]
test_sentences  = sentences[TRAIN_SIZE + DEV_SIZE : TRAIN_SIZE + DEV_SIZE + TEST_SIZE]

print(f"Training set   : {len(train_sentences):>10,} sentences")
print(f"Development set: {len(dev_sentences):>10,} sentences")
print(f"Test set       : {len(test_sentences):>10,} sentences")

del raw_sentences, table, sentences

Training set   :    998,000 sentences
Development set:      1,000 sentences
Test set       :      1,000 sentences


---
## 3. Build Vocabulary & N-gram Counts

In [3]:
def pad_sentence(tokens, n):
    """Pad a sentence with (n-1) <s> tokens at the start and one </s> at the end."""
    return [SOS] * (n - 1) + tokens + [EOS]


def get_ngrams(tokens, n):
    """Extract all n-grams from a token list."""
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

In [4]:
# ── Build vocabulary ─────────────────────────────────────────────────────
vocab = set()
for sent in tqdm(train_sentences, desc="Building vocabulary"):
    vocab.update(sent)

vocab.add(SOS)
vocab.add(EOS)
vocab.add(UNK)

V = len(vocab)
print(f"Vocabulary size (V): {V:,}")

Building vocabulary: 100%|██████████| 998000/998000 [00:02<00:00, 415417.99it/s]

Vocabulary size (V): 670,753


In [5]:
# ── Count n-grams ────────────────────────────────────────────────────────
unigram_counts = Counter()

bigram_counts   = Counter()
bigram_context  = Counter()

trigram_counts   = Counter()
trigram_context  = Counter()

quadrigram_counts   = Counter()
quadrigram_context  = Counter()

for sent in tqdm(train_sentences, desc="Counting n-grams"):
    # Unigrams
    padded_1 = pad_sentence(sent, 1)
    for gram in get_ngrams(padded_1, 1):
        unigram_counts[gram] += 1
    
    # Bigrams
    padded_2 = pad_sentence(sent, 2)
    for gram in get_ngrams(padded_2, 2):
        bigram_counts[gram] += 1
        bigram_context[gram[:1]] += 1
    
    # Trigrams
    padded_3 = pad_sentence(sent, 3)
    for gram in get_ngrams(padded_3, 3):
        trigram_counts[gram] += 1
        trigram_context[gram[:2]] += 1
    
    # Quadrigrams
    padded_4 = pad_sentence(sent, 4)
    for gram in get_ngrams(padded_4, 4):
        quadrigram_counts[gram] += 1
        quadrigram_context[gram[:3]] += 1

N_unigrams = sum(unigram_counts.values())

print(f"\n{'Model':<12} {'Unique n-grams':>16} {'Total count':>14}")
print("-" * 46)
print(f"{'Unigram':<12} {len(unigram_counts):>16,} {N_unigrams:>14,}")
print(f"{'Bigram':<12} {len(bigram_counts):>16,} {sum(bigram_counts.values()):>14,}")
print(f"{'Trigram':<12} {len(trigram_counts):>16,} {sum(trigram_counts.values()):>14,}")
print(f"{'Quadrigram':<12} {len(quadrigram_counts):>16,} {sum(quadrigram_counts.values()):>14,}")

Counting n-grams: 100%|██████████| 998000/998000 [00:59<00:00, 16820.78it/s]



Model          Unique n-grams    Total count
----------------------------------------------
Unigram               670,751     11,439,087
Bigram              4,797,059     11,439,087
Trigram             7,831,299     11,439,087
Quadrigram          9,045,911     11,439,087


In [6]:
def unigram_prob_smoothed(word):
    """Laplace-smoothed unigram probability: P(w) = (C(w) + 1) / (N + V)"""
    count = unigram_counts.get((word,), 0)
    return (count + 1) / (N_unigrams + V)


def sentence_prob_unigram_smoothed(tokens):
    """Compute log probability of a sentence under the smoothed unigram model."""
    padded = pad_sentence(tokens, 1)
    log_prob = 0.0
    n_tokens = 0
    for w in padded:
        p = unigram_prob_smoothed(w)
        log_prob += math.log2(p)
        n_tokens += 1
    return log_prob, n_tokens


# ── Verify probabilities ─────────────────────────────────────────────────
# Show a few examples: known token vs. unknown token
sample_token = list(unigram_counts.keys())[0][0]
print(f"P_smoothed('{sample_token}') = {unigram_prob_smoothed(sample_token):.8f}")
print(f"P_smoothed('<UNK>')          = {unigram_prob_smoothed(UNK):.8f}")
print(f"P_smoothed('NEVER_SEEN_XYZ') = {unigram_prob_smoothed('NEVER_SEEN_XYZ'):.8f}")
print(f"\nAll probabilities are > 0 — no zero-probability issue!")

P_smoothed('పెట్రోల్') = 0.00008167
P_smoothed('<UNK>')          = 0.00000008
P_smoothed('NEVER_SEEN_XYZ') = 0.00000008

All probabilities are > 0 — no zero-probability issue!


In [7]:
def bigram_prob_smoothed(w_prev, w_curr):
    """Laplace-smoothed bigram: P(w_curr | w_prev) = (C(w_prev, w_curr) + 1) / (C(w_prev) + V)"""
    numerator   = bigram_counts.get((w_prev, w_curr), 0) + 1
    denominator = bigram_context.get((w_prev,), 0) + V
    return numerator / denominator


def sentence_prob_bigram_smoothed(tokens):
    """Compute log probability under the smoothed bigram model."""
    padded = pad_sentence(tokens, 2)
    log_prob = 0.0
    n_tokens = 0
    for i in range(1, len(padded)):
        p = bigram_prob_smoothed(padded[i-1], padded[i])
        log_prob += math.log2(p)
        n_tokens += 1
    return log_prob, n_tokens


# ── Show sample conditional probabilities ────────────────────────────────
print("Sample smoothed bigram probabilities:\n")
top_bigrams = bigram_counts.most_common(5)
for (w1, w2), count in top_bigrams:
    p_smooth = bigram_prob_smoothed(w1, w2)
    p_raw    = count / bigram_context.get((w1,), 1)
    print(f"  P({w2} | {w1})  raw={p_raw:.6f}  smoothed={p_smooth:.6f}")

Sample smoothed bigram probabilities:

  P(</s> | .)  raw=0.991595  smoothed=0.569018
  P(ఈ | <s>)  raw=0.051941  smoothed=0.031064
  P(. | ఉంది)  raw=0.866796  smoothed=0.036370
  P(. | చేశారు)  raw=0.954310  smoothed=0.030886
  P(</s> | ?)  raw=0.913245  smoothed=0.027934


In [8]:
def trigram_prob_smoothed(w_2, w_1, w_0):
    """Laplace-smoothed trigram: P(w_0 | w_2, w_1) = (C(w_2, w_1, w_0) + 1) / (C(w_2, w_1) + V)"""
    numerator   = trigram_counts.get((w_2, w_1, w_0), 0) + 1
    denominator = trigram_context.get((w_2, w_1), 0) + V
    return numerator / denominator


def sentence_prob_trigram_smoothed(tokens):
    """Compute log probability under the smoothed trigram model."""
    padded = pad_sentence(tokens, 3)
    log_prob = 0.0
    n_tokens = 0
    for i in range(2, len(padded)):
        p = trigram_prob_smoothed(padded[i-2], padded[i-1], padded[i])
        log_prob += math.log2(p)
        n_tokens += 1
    return log_prob, n_tokens


# ── Show sample conditional probabilities ────────────────────────────────
print("Sample smoothed trigram probabilities:\n")
top_trigrams = trigram_counts.most_common(5)
for (w1, w2, w3), count in top_trigrams:
    p_smooth = trigram_prob_smoothed(w1, w2, w3)
    p_raw    = count / trigram_context.get((w1, w2), 1)
    print(f"  P({w3} | {w1}, {w2})  raw={p_raw:.6f}  smoothed={p_smooth:.6f}")

Sample smoothed trigram probabilities:

  P(ఈ | <s>, <s>)  raw=0.051941  smoothed=0.031064
  P(</s> | ఉంది, .)  raw=0.998311  smoothed=0.036513
  P(</s> | చేశారు, .)  raw=0.999113  smoothed=0.030905
  P(</s> | ఉన్నాయి, .)  raw=0.999169  smoothed=0.027896
  P(అయితే | <s>, <s>)  raw=0.018929  smoothed=0.011321


In [9]:
def quadrigram_prob_smoothed(w_3, w_2, w_1, w_0):
    """Laplace-smoothed quadrigram: P(w_0 | w_3, w_2, w_1) = (C(w_3, w_2, w_1, w_0) + 1) / (C(w_3, w_2, w_1) + V)"""
    numerator   = quadrigram_counts.get((w_3, w_2, w_1, w_0), 0) + 1
    denominator = quadrigram_context.get((w_3, w_2, w_1), 0) + V
    return numerator / denominator


def sentence_prob_quadrigram_smoothed(tokens):
    """Compute log probability under the smoothed quadrigram model."""
    padded = pad_sentence(tokens, 4)
    log_prob = 0.0
    n_tokens = 0
    for i in range(3, len(padded)):
        p = quadrigram_prob_smoothed(padded[i-3], padded[i-2], padded[i-1], padded[i])
        log_prob += math.log2(p)
        n_tokens += 1
    return log_prob, n_tokens


# ── Show sample conditional probabilities ────────────────────────────────
print("Sample smoothed quadrigram probabilities:\n")
top_quadrigrams = quadrigram_counts.most_common(5)
for (w1, w2, w3, w4), count in top_quadrigrams:
    p_smooth = quadrigram_prob_smoothed(w1, w2, w3, w4)
    p_raw    = count / quadrigram_context.get((w1, w2, w3), 1)
    print(f"  P({w4} | {w1}, {w2}, {w3})  raw={p_raw:.6f}  smoothed={p_smooth:.6f}")

Sample smoothed quadrigram probabilities:

  P(ఈ | <s>, <s>, <s>)  raw=0.051941  smoothed=0.031064
  P(అయితే | <s>, <s>, <s>)  raw=0.018929  smoothed=0.011321
  P(ఆ | <s>, <s>, <s>)  raw=0.014529  smoothed=0.008690
  P(దీంతో | <s>, <s>, <s>)  raw=0.010470  smoothed=0.006262
  P(కానీ | <s>, <s>, <s>)  raw=0.010245  smoothed=0.006128


---
## 9. Evaluation — Perplexity (Smoothed Models)

In [10]:
def compute_perplexity(data_sentences, sentence_prob_fn):
    """
    Compute perplexity of a dataset under a given language model.
    
    With Laplace smoothing, no token has zero probability,
    so perplexity will always be finite.
    """
    total_log_prob = 0.0
    total_tokens   = 0
    
    for sent in data_sentences:
        lp, nt = sentence_prob_fn(sent)
        if math.isinf(lp):
            return float('inf')
        total_log_prob += lp
        total_tokens   += nt
    
    avg_log_prob = total_log_prob / total_tokens
    perplexity   = 2 ** (-avg_log_prob)
    return perplexity

In [11]:
# ── Compute perplexity for all 4 smoothed models ─────────────────────────
smoothed_models = {
    "Unigram":    sentence_prob_unigram_smoothed,
    "Bigram":     sentence_prob_bigram_smoothed,
    "Trigram":    sentence_prob_trigram_smoothed,
    "Quadrigram": sentence_prob_quadrigram_smoothed,
}

smoothed_results = {}

print("Perplexity with Laplace (Add-1) Smoothing:\n")
print(f"{'Model':<12} {'Dev PP':>15} {'Test PP':>15}")
print("-" * 45)

for name, prob_fn in smoothed_models.items():
    pp_dev  = compute_perplexity(dev_sentences, prob_fn)
    pp_test = compute_perplexity(test_sentences, prob_fn)
    smoothed_results[name] = {"dev": pp_dev, "test": pp_test}
    
    dev_str  = f"{pp_dev:,.2f}" if not math.isinf(pp_dev) else "inf"
    test_str = f"{pp_test:,.2f}" if not math.isinf(pp_test) else "inf"
    print(f"{name:<12} {dev_str:>15} {test_str:>15}")

Perplexity with Laplace (Add-1) Smoothing:

Model                 Dev PP         Test PP
---------------------------------------------
Unigram             6,384.81        6,386.94
Bigram             30,945.36       32,270.31
Trigram           167,963.14      173,218.43
Quadrigram        276,059.03      284,090.22


---
## 10. Comparison: Unsmoothed vs. Smoothed

To compare, we also compute the **unsmoothed** (MLE) perplexity side by side.

In [12]:
# ── Unsmoothed (MLE) probability functions ───────────────────────────────

def unigram_prob_raw(word):
    return unigram_counts.get((word,), 0) / N_unigrams

def sentence_prob_unigram_raw(tokens):
    padded = pad_sentence(tokens, 1)
    log_prob, n_tokens = 0.0, 0
    for w in padded:
        p = unigram_prob_raw(w)
        log_prob += math.log2(p) if p > 0 else float('-inf')
        n_tokens += 1
    return log_prob, n_tokens


def bigram_prob_raw(w_prev, w_curr):
    num = bigram_counts.get((w_prev, w_curr), 0)
    den = bigram_context.get((w_prev,), 0)
    return num / den if den > 0 else 0.0

def sentence_prob_bigram_raw(tokens):
    padded = pad_sentence(tokens, 2)
    log_prob, n_tokens = 0.0, 0
    for i in range(1, len(padded)):
        p = bigram_prob_raw(padded[i-1], padded[i])
        log_prob += math.log2(p) if p > 0 else float('-inf')
        n_tokens += 1
    return log_prob, n_tokens


def trigram_prob_raw(w_2, w_1, w_0):
    num = trigram_counts.get((w_2, w_1, w_0), 0)
    den = trigram_context.get((w_2, w_1), 0)
    return num / den if den > 0 else 0.0

def sentence_prob_trigram_raw(tokens):
    padded = pad_sentence(tokens, 3)
    log_prob, n_tokens = 0.0, 0
    for i in range(2, len(padded)):
        p = trigram_prob_raw(padded[i-2], padded[i-1], padded[i])
        log_prob += math.log2(p) if p > 0 else float('-inf')
        n_tokens += 1
    return log_prob, n_tokens


def quadrigram_prob_raw(w_3, w_2, w_1, w_0):
    num = quadrigram_counts.get((w_3, w_2, w_1, w_0), 0)
    den = quadrigram_context.get((w_3, w_2, w_1), 0)
    return num / den if den > 0 else 0.0

def sentence_prob_quadrigram_raw(tokens):
    padded = pad_sentence(tokens, 4)
    log_prob, n_tokens = 0.0, 0
    for i in range(3, len(padded)):
        p = quadrigram_prob_raw(padded[i-3], padded[i-2], padded[i-1], padded[i])
        log_prob += math.log2(p) if p > 0 else float('-inf')
        n_tokens += 1
    return log_prob, n_tokens

In [13]:
# ── Compute unsmoothed perplexity ────────────────────────────────────────
raw_models = {
    "Unigram":    sentence_prob_unigram_raw,
    "Bigram":     sentence_prob_bigram_raw,
    "Trigram":    sentence_prob_trigram_raw,
    "Quadrigram": sentence_prob_quadrigram_raw,
}

raw_results = {}
for name, prob_fn in raw_models.items():
    pp_dev  = compute_perplexity(dev_sentences, prob_fn)
    pp_test = compute_perplexity(test_sentences, prob_fn)
    raw_results[name] = {"dev": pp_dev, "test": pp_test}

In [14]:
# ── Side-by-side comparison table ────────────────────────────────────────
def fmt_pp(val):
    return f"{val:,.2f}" if not math.isinf(val) else "inf"

print("=" * 90)
print("COMPARISON: Unsmoothed (MLE) vs. Laplace-Smoothed Perplexity")
print("=" * 90)
print()
print(f"{'Model':<12} | {'Dev PP (MLE)':>15} {'Dev PP (Lap)':>15} | {'Test PP (MLE)':>15} {'Test PP (Lap)':>15}")
print("-" * 12 + "-+-" + "-" * 31 + "-+-" + "-" * 31)

for name in ["Unigram", "Bigram", "Trigram", "Quadrigram"]:
    rd = raw_results[name]
    sd = smoothed_results[name]
    print(
        f"{name:<12} | "
        f"{fmt_pp(rd['dev']):>15} {fmt_pp(sd['dev']):>15} | "
        f"{fmt_pp(rd['test']):>15} {fmt_pp(sd['test']):>15}"
    )

print()
print("Key observations:")
print("  - MLE models produce inf perplexity when unseen n-grams appear in dev/test data.")
print("  - Laplace smoothing assigns non-zero probability to ALL n-grams, yielding finite perplexity.")
print("  - Higher-order models (trigram, quadrigram) tend to have higher smoothed perplexity")
print("    because the probability mass is spread across V possible continuations.")

COMPARISON: Unsmoothed (MLE) vs. Laplace-Smoothed Perplexity

Model        |    Dev PP (MLE)    Dev PP (Lap) |   Test PP (MLE)   Test PP (Lap)
-------------+---------------------------------+--------------------------------
Unigram      |             inf        6,384.81 |             inf        6,386.94
Bigram       |             inf       30,945.36 |             inf       32,270.31
Trigram      |             inf      167,963.14 |             inf      173,218.43
Quadrigram   |             inf      276,059.03 |             inf      284,090.22

Key observations:
  - MLE models produce inf perplexity when unseen n-grams appear in dev/test data.
  - Laplace smoothing assigns non-zero probability to ALL n-grams, yielding finite perplexity.
  - Higher-order models (trigram, quadrigram) tend to have higher smoothed perplexity
    because the probability mass is spread across V possible continuations.


---
## 11. Summary

In [15]:
print("=" * 80)
print("SUMMARY: Laplace (Add-1) Smoothed N-gram Language Models")
print("=" * 80)
print(f"\nCorpus           : IndicCorpV2 Telugu (tel_Telu)")
print(f"Total sentences  : {NUM_SENTENCES:,}")
print(f"Training set     : {TRAIN_SIZE:,} sentences")
print(f"Development set  : {DEV_SIZE:,} sentences")
print(f"Test set         : {TEST_SIZE:,} sentences")
print(f"Vocabulary size  : {V:,}")
print()
print("Smoothing method : Add-1 (Laplace)")
print("Formula          : P_Laplace(w_n | w_1,...,w_{n-1}) = (C(w_1,...,w_n) + 1) / (C(w_1,...,w_{n-1}) + V)")
print()
print(f"{'Model':<12} {'Dev PP (Smoothed)':>20} {'Test PP (Smoothed)':>20}")
print("-" * 56)
for name in ["Unigram", "Bigram", "Trigram", "Quadrigram"]:
    sd = smoothed_results[name]
    print(f"{name:<12} {fmt_pp(sd['dev']):>20} {fmt_pp(sd['test']):>20}")

print("\n" + "=" * 80)

SUMMARY: Laplace (Add-1) Smoothed N-gram Language Models

Corpus           : IndicCorpV2 Telugu (tel_Telu)
Total sentences  : 1,000,000
Training set     : 998,000 sentences
Development set  : 1,000 sentences
Test set         : 1,000 sentences
Vocabulary size  : 670,753

Smoothing method : Add-1 (Laplace)
Formula          : P_Laplace(w_n | w_1,...,w_{n-1}) = (C(w_1,...,w_n) + 1) / (C(w_1,...,w_{n-1}) + V)

Model           Dev PP (Smoothed)   Test PP (Smoothed)
--------------------------------------------------------
Unigram                  6,384.81             6,386.94
Bigram                  30,945.36            32,270.31
Trigram                167,963.14           173,218.43
Quadrigram             276,059.03           284,090.22

